# Notebook 05 — Zero-Shot LLM Batch Processing

Sends all 10,492 emails to Qwen2.5 3B for zero-shot three-class classification, generating confidence scores for each email.

## What This Notebook Does from Notebook 04
- Sends each email to Qwen2.5 3B via Ollama with a structured prompt
- Parses JSON response containing predicted label and confidence scores
- Auto-saves progress every 50 emails

## Prerequisites
- Ollama must be running locally
- Qwen2.5 3B must be downloaded: ollama pull qwen2.5:3b

## Inputs
- data/processed/dataset_final.csv (from Notebook 04)

## Outputs
- data/processed/llm_features.csv — zero-shot confidence scores for all 10,492 emails

## Key Finding
Zero-shot Qwen2.5 3B correctly identified only 36 of 2,492 
AI phishing emails (1.4% recall) — demonstrating that zero-shot classification is insufficient for AI phishing detection and motivating the fine-tuning approach.

## Runtime
Approximately 16 hours
Output file is already saved in data/processed/

In [ ]:
import requests
import json
import pandas as pd
import numpy as np
import time
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Paths
BASE_DIR = Path("C:/phishing_detection")
DATA_PROCESSED = BASE_DIR / "data" / "processed"

# Load finalised dataset
dataset = pd.read_csv(DATA_PROCESSED / "dataset_final.csv")
print(f"Dataset loaded: {len(dataset)} emails")
print(dataset['label'].value_counts().sort_index())

# Verify Ollama is running
try:
    response = requests.get("http://localhost:11434/api/tags")
    models = [m['name'] for m in response.json()['models']]
    print(f"\nOllama running. Models available:")
    for m in models:
        print(f"  - {m}")
except Exception as e:
    print(f"\nERROR: Ollama not running — {e}")
    print("Start it with:")
    print('"C:\\Users\\hp\\AppData\\Local\\Programs\\Ollama\\ollama.exe" serve')

In [ ]:
def classify_email(text, model="qwen2.5:3b", max_retries=3):
    """Send email to LLM and get confidence scores for all three classes."""
    
    # Truncate very long emails to save processing time
    if len(text) > 1000:
        text = text[:1000]
    
    prompt = f"""Classify this email into exactly one of three categories:
- legitimate: a normal, genuine email
- human_phishing: a phishing email written by a human
- ai_generated_phishing: a phishing email generated by an AI

Reply with ONLY a JSON object, no explanation:
{{"label": "legitimate", "conf_legitimate": 0.85, "conf_human_phishing": 0.10, "conf_ai_phishing": 0.05}}

The three confidence values must add up to 1.0.

Email:
{text}"""

    for attempt in range(max_retries):
        try:
            response = requests.post(
                "http://localhost:11434/api/generate",
                json={
                    "model": model,
                    "prompt": prompt,
                    "stream": False,
                    "options": {
                        "temperature": 0.1,  # Low temperature = consistent outputs
                        "max_tokens": 100
                    }
                },
                timeout=30
            )
            
            if response.status_code == 200:
                raw = response.json()["response"].strip()
                
                # Extract JSON from response
                start = raw.find('{')
                end = raw.rfind('}') + 1
                if start != -1 and end > start:
                    parsed = json.loads(raw[start:end])
                    return {
                        'llm_label': parsed.get('label', 'unknown'),
                        'conf_legitimate': float(parsed.get('conf_legitimate', 0.33)),
                        'conf_human_phishing': float(parsed.get('conf_human_phishing', 0.33)),
                        'conf_ai_phishing': float(parsed.get('conf_ai_phishing', 0.33))
                    }
        except Exception:
            if attempt < max_retries - 1:
                time.sleep(2)
            continue
    
    # Return neutral scores if all attempts fail
    return {
        'llm_label': 'unknown',
        'conf_legitimate': 0.33,
        'conf_human_phishing': 0.33,
        'conf_ai_phishing': 0.33
    }

# Test on 3 sample emails
print("Testing classifier on 3 sample emails...\n")
for i in [0, 4000, 8000]:
    sample = dataset.iloc[i]
    result = classify_email(sample['text'])
    print(f"Email {i} (true label: {sample['label']})")
    print(f"  LLM label: {result['llm_label']}")
    print(f"  Conf legitimate:      {result['conf_legitimate']:.2f}")
    print(f"  Conf human phishing:  {result['conf_human_phishing']:.2f}")
    print(f"  Conf AI phishing:     {result['conf_ai_phishing']:.2f}")
    print()

In [ ]:
save_path = DATA_PROCESSED / "llm_features.csv"

# Check for existing progress
if save_path.exists():
    existing = pd.read_csv(save_path)
    start_idx = len(existing)
    results = existing.to_dict('records')
    print(f"Resuming from email {start_idx}/{len(dataset)}")
else:
    start_idx = 0
    results = []
    print(f"Starting fresh — {len(dataset)} emails to process")

print(f"Estimated time remaining: {((len(dataset) - start_idx) * 5) / 3600:.1f} hours")

for idx in tqdm(range(start_idx, len(dataset)), desc="LLM Classification"):
    row = dataset.iloc[idx]
    result = classify_email(row['text'])
    result['email_idx'] = idx
    result['true_label'] = row['label']
    results.append(result)
    
    # Auto-save every 50 emails
    if (idx + 1) % 50 == 0:
        pd.DataFrame(results).to_csv(save_path, index=False)

# Final save
final_df = pd.DataFrame(results)
final_df.to_csv(save_path, index=False)

print(f"\nBATCH PROCESSING COMPLETE")
print(f"Total emails processed: {len(final_df)}")
print(f"\nLLM label distribution:")
print(final_df['llm_label'].value_counts())
print(f"\nSaved to: {save_path}")